# Inverse estimates and logarithmic maximum-norm errors

Let $\Omega=(0,1)^2$ and $V_h=S_1^0(\mathcal T_h)\cap H_0^1(\Omega)$. For continuous piecewise linear functions on a shape-regular, quasiuniform triangular mesh in two dimensions, inverse estimates give

$$
\begin{align*}
|v_h|_{H^1(\Omega)}\le C h^{-1}\|v_h\|_{L^2(\Omega)}, \qquad \|v_h\|_{L^\infty(\Omega)}\le C h^{-1}\|v_h\|_{L^2(\Omega)}.
\end{align*}
$$

The negative power of $h$ is the price of bounding it by the weaker $L^2$ norm on a discrete space.

We first see both inverse factors using a nodal hat. We then construct a smooth Poisson example where the Galerkin maximum error really contains $h^2\log(1/h)$.

In [ ]:
from ngsolve import *
from netgen.meshing import Mesh as NetgenMesh, MeshPoint, Pnt, Element2D, Element1D
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt

## 1. Same element shapes, different diagonal directions

Subdivide the unit square into $n\times n$ squares, with even $n$, and split each into two right isosceles triangles. We use the grid width $h=1/n$. The maximum element diameter is $\sqrt2\,h$.

The parallel pattern uses the same diagonal everywhere. In the second pattern, diagonals point away from the center $z=(1/2,1/2)$: use the southwest–northeast diagonal when the cell midpoint satisfies $(x-1/2)(y-1/2)<0$, and the other diagonal otherwise. Four triangles then meet at $z$, compared with six in the parallel pattern.

Both mesh families are uniformly shape regular and quasiuniform.

In [ ]:
def square_mesh(n, four_triangle_patch=False):
    if n < 2 or n % 2:
        raise ValueError("Use an even number of subdivisions.")
    ngmesh = NetgenMesh(dim=2)
    domain = ngmesh.AddRegion("domain", dim=2)
    boundary = ngmesh.AddRegion("boundary", dim=1)
    points = [[ngmesh.Add(MeshPoint(Pnt(i/n, j/n, 0)))
               for i in range(n+1)] for j in range(n+1)]

    for j in range(n):
        for i in range(n):
            bl, br = points[j][i], points[j][i+1]
            tl, tr = points[j+1][i], points[j+1][i+1]
            southwest_northeast = True
            if four_triangle_patch:
                southwest_northeast = (i+0.5-n/2)*(j+0.5-n/2) < 0
            triangles = ([bl, br, tr], [bl, tr, tl]) if southwest_northeast else (
                         [bl, br, tl], [br, tr, tl])
            for triangle in triangles:
                ngmesh.Add(Element2D(domain, triangle))

    for i in range(n):
        ngmesh.Add(Element1D([points[0][i], points[0][i+1]], index=boundary))
        ngmesh.Add(Element1D([points[n][i+1], points[n][i]], index=boundary))
    for j in range(n):
        ngmesh.Add(Element1D([points[j+1][0], points[j][0]], index=boundary))
        ngmesh.Add(Element1D([points[j][n], points[j+1][n]], index=boundary))
    return Mesh(ngmesh)


subdivisions = [8, 16, 32, 64, 128]
view_n = 16
sample_subdivisions = 8

Draw(square_mesh(8), name="Parallel diagonals")
Draw(square_mesh(8, four_triangle_patch=True), name="Four triangles at the center");

## 2. A nodal hat makes the inverse factors visible

Let $\phi_h$ be one at the center vertex and zero at every other vertex, on the parallel mesh. It is supported on a patch with area proportional to $h^2$.

Its height stays one, its $L^2$ norm is proportional to $h$, and its gradient is proportional to $h^{-1}$ on that patch. Consequently,

$$
\begin{align*}
\|\phi_h\|_\infty=1,\qquad \|\phi_h\|_{L^2}\asymp h,\qquad |\phi_h|_{H^1}\asymp1.
\end{align*}
$$

Thus both stronger-norm/$L^2$ ratios grow like $h^{-1}$. Multiplying the ratios by $h$ should remove their mesh dependence. For a $P_1$ function, the maximum of its absolute value is attained at a vertex.

In [ ]:
hat_h, hat_L2, hat_H1, hat_Linf = [], [], [], []

for n in subdivisions:
    mesh = square_mesh(n)
    fes = H1(mesh, order=1, dirichlet=".*")
    center_vertex = (n//2)*(n+1)+n//2
    center_dof = fes.GetDofNrs(NodeId(VERTEX, center_vertex))[0]
    hat = GridFunction(fes)
    hat.vec[center_dof] = 1

    hat_h.append(1/n)
    hat_L2.append(sqrt(Integrate(hat**2, mesh)))
    hat_H1.append(sqrt(Integrate(grad(hat)**2, mesh)))
    hat_Linf.append(np.max(abs(hat.vec.FV().NumPy())))

    if n == view_n:
        Draw(hat, mesh, name="A shrinking nodal hat", deformation=True)

hat_h = np.array(hat_h)
hat_L2, hat_H1, hat_Linf = map(np.array, [hat_L2, hat_H1, hat_Linf])
print(" n       L2 norm      H1 seminorm    maximum")
for n, e0, e1, em in zip(subdivisions, hat_L2, hat_H1, hat_Linf):
    print(f"{n:3d}    {e0:11.4e}    {e1:11.4e}    {em:7.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for values, label in [(hat_H1/hat_L2, r"$|\phi_h|_{H^1}/\|\phi_h\|_{L^2}$"),
                       (hat_Linf/hat_L2, r"$\|\phi_h\|_\infty/\|\phi_h\|_{L^2}$")]:
    axes[0].loglog(hat_h, values, "o-", label=label)
    axes[1].semilogx(hat_h, hat_h*values, "o-", label=label)
reference = (hat_H1/hat_L2)[-1]*hat_h[-1]/hat_h
axes[0].loglog(hat_h, reference, "k--", label=r"reference $h^{-1}$")
axes[0].set(xlabel=r"grid width $h=1/n$", ylabel="norm ratio",
            title="Stronger norms versus L2")
axes[1].set(xlabel=r"grid width $h=1/n$", ylabel=r"$h$ times the norm ratio",
            title="The rescaled ratios stay constant")
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=9)
plt.show()

The hat becomes narrower, not taller. Its $L^2$ norm tends to zero while its energy seminorm and maximum stay fixed. This rules out an $h$-independent bound for either ratio.

These are ratios for a particular family of functions. They demonstrate the necessary scaling.

## 3. A smooth solution with a logarithmic Galerkin error

Use the same fixed smooth function on both mesh families,

$$
\begin{align*}
u_{\rm ex}=x(1-x)y(1-y),\qquad -\Delta u_{\rm ex}=f=2\bigl[x(1-x)+y(1-y)\bigr]
\end{align*}
$$

on $(0,1)^2$, with zero Dirichlet data. We find $u_h\in V_h$ such that

$$
\begin{align*}
\int_\Omega\nabla u_h\cdot\nabla v_h\,dx =\int_\Omega f v_h\,dx \qquad\forall v_h\in V_h.
\end{align*}
$$

The elementary inverse-inequality argument gives an $O(h)$ maximum-error bound. For this smooth solution, the sharper bound is $O(h^2(1+|\log h|))$.

For the four-triangle patch, the logarithm is actually present. We measure both the error at $z$ and a sampled maximum error. The center value is an exact pointwise measurement, hence a lower bound for the full maximum error.

The error $u_{\rm ex}-u_h$ is not piecewise linear. Its maximum need not be at a vertex, so taking the largest nodal error alone would not compute its $L^\infty$ norm.

In [ ]:
u_ex = x*(1-x)*y*(1-y)
f = 2*(x*(1-x)+y*(1-y))


def galerkin_solution(mesh):
    fes = H1(mesh, order=1, dirichlet=".*")
    u, v = fes.TnT()
    a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
    # The quadratic source times a linear test function has degree three.
    load = LinearForm(f*v*dx(bonus_intorder=1)).Assemble()
    uh = GridFunction(fes)
    uh.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky")*load.vec
    interpolant = GridFunction(fes)
    interpolant.Set(u_ex, dual=True)
    return fes, uh, interpolant


def sampled_maximum(function, mesh, m=sample_subdivisions):
    points = [(i/m, j/m) for i in range(m+1) for j in range(m+1-i)]
    rule = IntegrationRule(points=points, weights=[1.0]*len(points))
    mapped_points = mesh.MapToAllElements(rule, VOL)
    values = np.asarray(function(mapped_points)).reshape(-1)
    return np.max(abs(values))

In [ ]:
hs = 1/np.array(subdivisions, dtype=float)
maximum_errors, center_errors, discrete_errors = {}, {}, {}
view_cases = []

for name, four_patch in [("parallel diagonals", False), ("four-triangle patch", True)]:
    maximum_errors[name], center_errors[name], discrete_errors[name] = [], [], []
    for n in subdivisions:
        mesh = square_mesh(n, four_triangle_patch=four_patch)
        fes, uh, interpolant = galerkin_solution(mesh)
        difference = interpolant.vec.CreateVector()
        difference.data = interpolant.vec-uh.vec
        center = mesh(0.5, 0.5)

        maximum_errors[name].append(sampled_maximum(u_ex-uh, mesh))
        center_errors[name].append(abs(u_ex(center)-uh(center)))
        discrete_errors[name].append(np.max(abs(difference.FV().NumPy())))

        if n == view_n:
            view_cases.append((name, mesh, uh, interpolant))

    maximum_errors[name] = np.array(maximum_errors[name])
    center_errors[name] = np.array(center_errors[name])
    discrete_errors[name] = np.array(discrete_errors[name])

print(" n      parallel: max/h²    four-patch: max/h²    four-patch: center/h²")
for k, n in enumerate(subdivisions):
    print(f'{n:3d}       {maximum_errors["parallel diagonals"][k]/hs[k]**2:10.6f}'
          f'             {maximum_errors["four-triangle patch"][k]/hs[k]**2:10.6f}'
          f'               {center_errors["four-triangle patch"][k]/hs[k]**2:10.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for name in maximum_errors:
    axes[0].loglog(hs, maximum_errors[name], "o-", label=name)
    axes[1].plot(np.log(1/hs), maximum_errors[name]/hs**2, "o-", label=name)

four_patch_error = maximum_errors["four-triangle patch"]
axes[0].loglog(hs, maximum_errors["parallel diagonals"][-1]*(hs/hs[-1])**2,
               "k:", label=r"reference $h^2$")
axes[0].loglog(hs, four_patch_error[-1]*(hs/hs[-1])**2
               *(1+np.log(1/hs))/(1+np.log(1/hs[-1])),
               "k--", label=r"reference $h^2(1+\log(1/h))$")
axes[1].plot(np.log(1/hs), center_errors["four-triangle patch"]/hs**2,
             "s--", label="four-patch center error")
axes[0].set(xlabel=r"grid width $h=1/n$", ylabel="sampled maximum error",
            title="Both errors converge faster than order h")
axes[1].set(xlabel=r"$\log(1/h)$", ylabel=r"error divided by $h^2$",
            title="The logarithmic factor becomes visible")
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)
plt.show()

A log–log error plot can make $h^2$ and $h^2\log(1/h)$ look almost alike. Dividing by $h^2$ separates them: the parallel mesh gives an approximately constant ratio, while the four-triangle patch gives growth approximately linear in $\log(1/h)$.

The reference lines are aligned at the finest mesh to compare shapes. The exact center measurements show the same logarithmic trend.

The sampled maximum uses a barycentric lattice including vertices and edge points. It is a numerical approximation from below to the true maximum. Increasing the sampling resolution below checks its effect on one representative mesh.

In [ ]:
for name, mesh, uh, interpolant in view_cases:
    discrete_difference = GridFunction(uh.space)
    discrete_difference.vec.data = interpolant.vec-uh.vec
    Draw(view_n**2*discrete_difference, mesh,
         name=name+" — scaled discrete error", deformation=True)

    if name == "four-triangle patch":
        coarse_sampling = sampled_maximum(u_ex-uh, mesh, m=8)
        fine_sampling = sampled_maximum(u_ex-uh, mesh, m=16)
        print(f"Maximum sampling, n={view_n}: m=8: {coarse_sampling:.6e}, "
              f"m=16: {fine_sampling:.6e}")
        print(f"Relative change: {abs(fine_sampling-coarse_sampling)/fine_sampling:.2%}")

print(" n-pair       parallel EOC       four-patch EOC")
for k in range(len(hs)-1):
    rates = [np.log(maximum_errors[name][k]/maximum_errors[name][k+1])
             /np.log(hs[k]/hs[k+1]) for name in maximum_errors]
    print(f"{subdivisions[k]:3d}->{subdivisions[k+1]:<3d}       {rates[0]:8.4f}            {rates[1]:8.4f}")

## Observe

- Why do both norm ratios of the shrinking hat grow like $h^{-1}$?
- Both mesh families have the same triangle shapes. What does dividing the maximum error by $h^2$ reveal?

**Takeaway.** The inverse estimates describe mesh-dependent norm ratios of discrete functions. The maximum-error experiment concerns the Galerkin approximation of a fixed smooth solution. On the four-triangle patches, the normalized errors exhibit logarithmic growth. This factor is compatible with an EOC below two that approaches two slowly.

[← Corner singularity](03_corner_singularity.ipynb)

[Lecture notebooks](../README.md) · [Lecture notes](../../script/README.md) · [Course overview](../../index.ipynb)